*Tree of Thought*

In [1]:
import ollama

In [2]:
# Chain-of-thought -- single path

response = ollama.chat(
    model = "qwen3:0.6b",
    messages = [
        {
            "role": "user",
            "content": "A school needs to choose a new sport for their annual day. They have a budget of Rs. 50,000, 200 students, and an outdoor field. Which sport should they choose and why? Think step by step. /no_think"
        }
    ]
)

print(response['message']['content'])

To determine the best sport for the school, we need to evaluate the budget, the number of students, and the available outdoor field.

1. **Budget**: The school has a total budget of Rs. 50,000, and they need to choose a sport for 200 students. If we divide the total budget by the number of students, each student would need to pay **Rs. 250** (50,000 / 200 = 250). This ensures the sport is affordable for all participants.

2. **Outdoor Field Availability**: The problem explicitly states that the school has an outdoor field available, which is crucial for any sport. This ensures the sport can be played in the field, which is ideal for a school setting.

3. **Cost Per Student**: The cost per student is Rs. 250, which is consistent across all sports, making it a feasible option for 200 students.

4. **Sports Consideration**: Among the options, **soccer** is a popular and cost-effective sport that fits within the budget and can be played in the outdoor field. Other sports, such as basketbal

In [3]:
# Tree-of-thought -- multiple paths, compare, pick best

response = ollama.chat(
    model="qwen3:0.6b",
    messages=[
        {
            "role": "system",
            "content": "You solve problems by exploring multiple approaches. For each approach, evaluate its pros and cons. Then pick the best one."
        },
        {
            "role": "user",
            "content": """A school needs to choose a new sport for their annual day. They have a budget of Rs. 50,000, 200 students, and an outdoor field.
            Explore exactly 3 possible sports. For each sport:
            1. Name the sport
            2. List 2 pros (advantages)
            3. List 2 cons (disadvantages)
            4. Estimate the cost
            After exploring all 3, pick the BEST option and explain why. /no_think"""
        }
    ]
)
print(response["message"]["content"])

After exploring all three sports, the **BEST option is football**. Here's why:

1. **Sport**: Football  
   - **Pros**:  
     - Popular and easy to play, with a strong team spirit.  
     - Requires physical training, making it a good fit for a school with 200 students.  
   - **Cons**:  
     - High cost to train and maintain.  
     - Potential for injuries due to physical demands.  
   - **Cost**: Rs. 20,000.  

2. **Comparison with Other Options**:  
   - **Basketball**:  
     - Pros: Popular and has a good number of players.  
     - Cons: Expensive and risky for injuries.  
     - Cost: Rs. 15,000.  
   - **Soccer**:  
     - Pros: Team-oriented, has a good number of players.  
     - Cons: Physical demands and injury risk.  
     - Cost: Rs. 12,000.  

3. **Best Option**:  
   Football fits within the budget and offers the highest number of players and team spirit, making it the optimal choice for a school with 200 students and an outdoor field. Its cost is within Rs. 50,000, 

*ReAct Pattern -- Reason + Act*

In [4]:
response = ollama.chat(
    model="qwen3:0.6b",
    messages=[
        {
            "role": "system",
            "content": """You are a helpful assistant that solves problems step by step.When you need information you do not have, follow this pattern:
            Thought: [what you are thinking and what you need to find out]
            Action: [what tool you would use -- calculator, search, database]
            Observation: [you will be given the result]
            ... repeat Thought/Action/Observation if needed ...
            Final Answer: [your answer based on all observations]
            Always start with a Thought."""
        },
        {
            "role": "user",
            "content": "I want to buy 3 shirts at Rs. 899 each and 2 pants at Rs. 1,499 each. What is the total cost, and how much GST (18%) do I pay on top? /no_think"
        }
    ]
)
print(response["message"]["content"])

Thought: To calculate the total cost, first sum the cost of the shirts and pants. Then, calculate the GST on the total.  
Action: Use calculator to sum the shirt costs and pant costs.  
Observation: The total cost of the shirts and pants is Rs. 6,695.  
Action: Calculate GST on the total cost.  
Observation: GST amount is Rs. 125.10.  
Final Answer: The total cost is Rs. 6,695, and GST is Rs. 125.10.


In [32]:
response = ollama.chat(
    model="qwen3:0.6b",
    messages=[
        {
            'role':'system',
            'content':"""You are a travel planning assistant. Use ReAct pattern:
            Thought:[What you need to figure out next]
            Action:[what you would lookup -- flights,hotel,weather,distances]
            Observation:[Provide a realistic estimate if you donot have read data]
            .....repeat if needed......
            Final Answer:[your complete recommendation]
            Always show your reasoning. Always start with Thought"""
        },
        {
            'role':'user',
            'content':'Plan a 2 day weekend trip from Bangalore to Goa for 2 people.Budget is Rs20,000 total should we fly or take bus?'
        }
    ]
)

print(response['message']['content'])

Thought:  
To plan a 2-day trip from Bangalore to Goa with a budget of Rs20,000, we need to evaluate the cost of both options.  

**Flights:**  
- Flight from Bangalore to Goa (round trip): Rs800 (approx.).  
- Accommodation: Rs1,000 (2 nights).  
- Activities: Rs500.  
**Total cost:** Rs1,350.  

**Bus:**  
- Bus cost from Bangalore to Goa: Rs1,000 (approx.).  
- Accommodation: Rs500.  
**Total cost:** Rs1,500.  

**Recommendation:**  
Given the budget, flying (Rs800) is more economical, while taking a bus (Rs1,000) is also feasible. The best option is to fly from Bangalore to Goa, then plan for the two days with a balance of accommodation, activities, and transportation.  

Final Answer:  
To plan the trip, choose to fly from Bangalore to Goa. Flight cost is within the budget, and for the two days, plan to book a 2-night stay with accommodation, activities, and transportation. A total cost of Rs20,000 is feasible.


*Function Calling*

In [6]:
# Step 1: Define the Function

def calculate_discount(price, discount_percent):
    """Calculate the discounted price of a product.
    Args:
    price: The original price in Rs.
    discount_percent: The discount percentage (e.g., 20 for 20%)
    Returns:
    A dictionary with original price, discount amount, and final price.
    """
    discount_amount = price * (discount_percent / 100)
    final_price = price - discount_amount
    return {
        "original_price": price,
        "discount_percent": discount_percent,
        "discount_amount": discount_amount,
        "final_price": final_price
    }

In [7]:
import json

In [15]:
# Step 2: Describe the Function to the Model
# Define the tool description -- this tells the model what the function does

tools=[
    {
        'type':'function',
        'function':{
            'name':'calculate_discount',
            'description':'Calculate the discounted price of product,use this when someone asks about discounts or sale price',
            'parameters':{
                'type':'object',
                'properties':{
                    'price':{
                        'type':'number',
                        'description':'the original price of the product in Rs'
                    },
                    'discount_percent':{
                        'type':'number',
                        'description':'The discount percentage eg 20 for 20%'
                    }
                },
                'required':['price','discount_percent']
            }
        }
    }
]

print('Tool registered')
print(json.dumps(tools,indent=2))

Tool registered
[
  {
    "type": "function",
    "function": {
      "name": "calculate_discount",
      "description": "Calculate the discounted price of product,use this when someone asks about discounts or sale price",
      "parameters": {
        "type": "object",
        "properties": {
          "price": {
            "type": "number",
            "description": "the original price of the product in Rs"
          },
          "discount_percent": {
            "type": "number",
            "description": "The discount percentage eg 20 for 20%"
          }
        },
        "required": [
          "price",
          "discount_percent"
        ]
      }
    }
  }
]


In [21]:
# Step 3: Send a Message with Tools

# NOTE: Do NOT use /no_think with tool calling -- the model needs to think about which tool to use

response=ollama.chat(
    model='qwen3:0.6b',
    messages=[
        {
            'role':'system',
            'content':'You have access to tools when the user asks a calculation question, use the available tools instead of calculating yourself.'
        },
        {
            'role':'user',
            'content':"I want to buy a laptop that cost Rs65000. there is an 15% flipkart sale.What will I pay"
        }
    ],

    tools=tools
)

# Inspect the complete response first
print(response)

# show the tool call the model made
# The model returns a tool call, NOT a text answer

if response['message'].get('tool_calls'):
    tc=response['message']['tool_calls'][0]
    print("Model request a tool call!")
    print(f"function: {tc['function']['name']}")
    print(f"arguments: {tc['function']['arguments']}")

else:
    print("Model directly(no tool call)")
    print(response['message']['content'])

model='qwen3:0.6b' created_at='2026-10-05T18:59:25.610639Z' done=True done_reason='stop' total_duration=9525487500 load_duration=14023700 prompt_eval_count=227 prompt_eval_duration=59355000 eval_count=175 eval_duration=9440098000 message=Message(role='assistant', content='', thinking='Okay, the user wants to know how much they\'ll pay for a laptop that costs Rs65,000 with an 15% discount. Let me think. The tools provided include a function called calculate_discount, which takes discount_percent and price. The user provided both values: discount is 15%, price is 65000. So I need to use that function with those parameters. The function should return the discounted price. Let me check the parameters again. The discount is 15%, so the arguments should be {"discount_percent": 15, "price": 65000}. Then, call the function with those values.\n', images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='calculate_discount', arguments={'discount_percent': 15, 'price': 65000}))]) 

In [24]:
# Step 4: Run the Function and Send Result Back
# check if model wants to call a function

if response["message"].get("tool_calls"):
    tool_call = response["message"]["tool_calls"][0]
    function_name = tool_call["function"]["name"]
    arguments = tool_call["function"]["arguments"]

    print(f"Model wants to call: {function_name}")
    print(f"With arguments: {arguments}")

    # Run the function
    result = calculate_discount(**arguments)
    print(f"Functional result: {result}")
    # Send result back to the model
    final_response = ollama.chat(
        model="qwen3:0.6b",
        messages=[
            {
                'role':'system',
                'content':'You have access to tools when the user asks a calculation question, use the available tools instead of calculating yourself'
            },
            {
                'role':'user',
                'content':"I want to buy a laptop that cost Rs65000. there is an 15% flipkart sale.What will I pay"
            },
            response["message"],
            {
                "role": "tool", 
                "content": json.dumps(result)
            }
        ],
    )

    print(final_response["message"]["content"])

else:
    print("Model directly answer no tool call")
    print(response['message']['content'])

Model wants to call: calculate_discount
With arguments: {'discount_percent': 15, 'price': 65000}
Functional result: {'original_price': 65000, 'discount_percent': 15, 'discount_amount': 9750.0, 'final_price': 55250.0}
After applying the 15% discount, you will pay **Rs 55,250** for the laptop.


*Multiple Tool*

In [47]:
# define another function

def get_product_info(product_name):
    """Look at product information from database
    Args:
    product name:The name of the product to look up
    Returns:
    A dictionary with product details"""

    # We just use an dictionary to store information in real world we will define sql querry

    products ={
        'i phone 15':{'name':'I Phone 15', 'price':79999, 'brand':'Apple','rating':4.5},
        'samsung galaxy s24':{'name':'Samsung Galaxy S24', 'price':74999, 'brand':'Samsung', 'rating':4.6},
        'one plus 12':{'name':'One Plus 12', 'price':64990, 'brand':'One Plus', 'rating':4.2}
    }

    key = product_name.lower()

    if key in products:
        return products[key]

    return {'error':f"Product '{product_name}' not found"}

In [48]:
# testing

print(get_product_info('samsung galaxy s24'))

{'name': 'Samsung Galaxy S24', 'price': 74999, 'brand': 'Samsung', 'rating': 4.6}


In [34]:
tools_list=[
    {
        'type':'function',
        'function':{
            'name':'calculate_discount',
            'description':'Calculate the discounted price of product,use this when someone asks about discounts or sale price',
            'parameters':{
                'type':'object',
                'properties':{
                    'price':{
                        'type':'number',
                        'description':'the original price of the product in Rs'
                    },
                    'discount_percent':{
                        'type':'number',
                        'description':'The discount percentage eg 20 for 20%'
                    }
                },
                'required':['price','discount_percent']
            }
        }
    },
    {
        'type':'function',
        'function':{
            'name':'get_product_info',
            'description':'Look up product details like price brand and rating.Use when someone asks about a specific products',
            'parameters':{
                'type':'object',
                'properties':{
                    'product_name':{
                        'type':'string',
                        'describe':'Name of product to look up'
                    },
                },
                'required':['product_name']
            }
        }
    }
]

In [49]:
# new model
response = ollama.chat(
    model = 'qwen3:0.6b',
    messages = [
        {
            'role':'system',
            'content':'You have access to tools. ALWAYS use the available tools to answer the question',
        },
        {
            'role':'user',
            'content':'What is the price of Samsung Galaxy S24?'
        }
    ],
    tools = tools_list
)

In [50]:
# check if model call the tool

if response['message'].get('tool_calls'):
    tc=response['message']['tool_calls'][0]
    print("Model request a tool call!")
    print(f"function: {tc['function']['name']}")
    print(f"arguments: {tc['function']['arguments']}")

else:
    print("Model directly(no tool call)")
    print(response['message']['content'])

Model request a tool call!
function: get_product_info
arguments: {'product_name': 'Samsung Galaxy S24'}


In [51]:
# Create the available_functions dictionary
available_functions = {
    "calculate_discount": calculate_discount,
    "get_product_info": get_product_info,
}

In [52]:
# Step 4: Run the Function and Send Result Back
# check if model wants to call a function

if response["message"].get("tool_calls"):
    for tool_call in response['message']['tool_calls']:
        function_name = tool_call["function"]["name"]
        arguments = tool_call["function"]["arguments"]

        print(f"Model wants to call: {function_name}")
        print(f"With arguments: {arguments}")

        # Run the function
        function = available_functions.get(function_name)
        if function is not None:
            result = function(**arguments)

        print(f"Functional result: {result}")
        # Send result back to the model
        final_response = ollama.chat(
            model="qwen3:0.6b",
            messages=[
                {
                    'role':'system',
                    'content':'You have access t tools. ALWAYS use the available tools to answer the question'
                },
                {
                    'role':'user',
                    'content':'What is the price of Samsung Galaxy S24?'
                },
                response["message"],
                {
                    "role": "tool", 
                    "content": json.dumps(result)
                }
            ],
        )

    print(final_response["message"]["content"])

else:
    print("Model directly answer no tool call")
    print(response['message']['content'])

Model wants to call: get_product_info
With arguments: {'product_name': 'Samsung Galaxy S24'}
Functional result: {'name': 'Samsung Galaxy S24', 'price': 74999, 'brand': 'Samsung', 'rating': 4.6}
The Samsung Galaxy S24 has a price of **74,999**. It's a Samsung phone with a rating of 4.6 out of 5 stars. Let me know if you need more details!


In [54]:
%pip install pydantic

Note: you may need to restart the kernel to use updated packages.


In [59]:
from pydantic import BaseModel, ValidationError

class ProductInfo(BaseModel):
    """Defines the expected shape of product info from the LLM."""
    name: str
    price: float
    category: str
    in_stock: bool

In [63]:
response = ollama.chat(
    model = 'qwen3:0.6b',
    messages = [
        {
            'role':'system',
            'content':'Extract product info as JSON wit keys: name(string), price(number), category(string) and In stock(boolean).',
        },
        {
            'role':'user',
            'content':'The samsung Galaxy Buds 3 are wireless earbuds available for Rs14999. They are currently in stock.'
        }
    ],
    format = 'json',
)

raw_json = response['message']['content']

print("Raw LLM Output:")
print(raw_json)

Raw LLM Output:
{ "name": "Samsung Galaxy Buds 3", "price": 14999, "category": "wireless earbuds", "in_stock": true }


In [64]:
# validation with pydantic
try:
    product = ProductInfo.model_validate_json(raw_json)
    print(f"Name: {product.name}")
    print(f"Price: {product.price}")
except ValidationError as e:
    print(f"Validation FAILED: {e}")

Name: Samsung Galaxy Buds 3
Price: 14999.0


*Prompt Versioning*

In [65]:
PROMPTS = {
    'product_extractor':{
        'version':'1.0',
        'system':'Extract product info as JSON with keys : name, price, category. Return only the JSON.',
        'description':'Extract product information from text',
    }
}

prompt = PROMPTS['product_extractor']

print(f"Using prompt version: {prompt['version']}")
print(f"Deacription: {prompt['description']}")
print()

response = ollama.chat(
    model = 'qwen3:0.6b',
    messages = [
        {
            'role':'system',
            'content':prompt['system'],
        },
        {
            'role':'user',
            'content':'The sony WH-1000XM5 headphone cost Rs29,999 /no_think'
        }
    ],
    format = 'json',
)

print(json.loads(response['message']['content']))

Using prompt version: 1.0
Deacription: Extract product information from text

{'name': 'Sony WH-1000XM5', 'price': '29999', 'category': 'headphones'}


In [67]:
PROMPTS = {
    'product_extractor_v1':{
        'version':'1.0',
        'system':'Extract product info as JSON with keys : name, price, category. Return only the JSON.',
        'description':'Extract product information from text',
    },
    'product_extractor_v2':{
        'version':'2.0',
        'system':'Extract product info as JSON with keys: name, price(as a number without currency symbol), category, brand. Return only the JSON.',
    }
}

test_input = "The Apple Airpod pro 2 is available for Rs 24,900 on Amazone. /no_think"

for prompt_key in['product_extractor_v1','product_extractor_v2']:
    prompt = PROMPTS[prompt_key]

    response = ollama.chat(
        model = 'qwen3:0.6b',
        messages = [
            {
                'role':'system',
                'content':prompt['system'],
            },
            {
                'role':'user',
                'content':test_input,
            }
        ],
        format = 'json',
    )

    print(f"Using prompt version: {prompt['version']}")
    print(json.loads(response['message']['content']))
    print()

Using prompt version: 1.0
{'name': 'Apple Airpod Pro 2', 'price': 24900, 'category': 'electronics'}

Using prompt version: 2.0
{'name': 'Apple Airpod Pro 2', 'price': 24900, 'category': 'electronics', 'brand': 'Apple'}



*Multi-Turn Function Calling*

In [79]:
# Reuse functions and tools_list
# map function name to actual function so we can look them by name

available_functions = {
    "calculate_discount": calculate_discount,
    "get_product_info": get_product_info,
}

# start conversation with history
history = [
    {
        "role": "system", 
        "content": "You are a shopping assistant. Use the available tools when needed."
    }
]

In [80]:
history.append({
    "role":"user",
    "content":"What is the price of i phone 15."
})

In [84]:
r = ollama.chat(
    model = "qwen3:0.6b",
    messages = history,
    tools = tools_list
)
print(r)

model='qwen3:0.6b' created_at='2026-10-06T19:50:33.323442Z' done=True done_reason='stop' total_duration=9723965500 load_duration=21192400 prompt_eval_count=252 prompt_eval_duration=75912000 eval_count=170 eval_duration=9568430000 message=Message(role='assistant', content='', thinking='Okay, the user is asking for the price of the iPhone 15. Let me check the tools provided. There\'s a function called get_product_info that looks up product details like price, brand, and rating. Since the user is specifically asking about the price, I should use this function. I need to call get_product_info with the product_name set to "i phone 15". Wait, but the function\'s parameters are [product_name], so I\'ll structure the tool call accordingly. No other functions are needed here. Let me make sure there\'s no other tool required. Nope, only get_product_info. So the correct tool call is to that function with the product name.\n', images=None, tool_name=None, tool_calls=[ToolCall(function=Function(nam

In [85]:
if r['message'].get('tool_calls'):
    tc = r['message']['tool_calls'][0]
    result = available_functions[tc['function']['name']](**tc['function']['arguments'])

    print(f"[Tool: {tc['function']['name']}] -> {result}")

    history.append(r['message'])
    history.append({'role':'tool','content':json.dumps(result)})

    final = ollama.chat(
        model='qwen3:0.6b',
        messages=history,
    )

    print(f"Q1 Answer: {final['message']['content']} \n")
    history.append({'role':'assistant','content':final['message']['content']})

[Tool: get_product_info] -> {'name': 'I Phone 15', 'price': 79999, 'brand': 'Apple', 'rating': 4.5}
Q1 Answer: The iPhone 15 is priced at **$79,999**. It's a great choice with a rating of 4.5 out of 5 stars. Let me know if you need further details! 🎉 

